# <center> TD Data Mining — DATA HUNTER CHALLENGE

**Objectif :** collecter des données issues de trois sources Web différentes, les structurer dans des DataFrames Pandas, les enregistrer en CSV, puis comparer les méthodes.

| Mission | Méthode | Source | Fichier |
|---|---|---|---|
| 1 — Static Web Scraping | Requests + BeautifulSoup | books.toscrape.com | `books.csv` |
| 2 — API Explorer | Requests + API REST | openlibrary.org | `openlibrary.csv` |
| 3 — Dynamic Web Scraping | Selenium | quotes.toscrape.com/js | `quotes.csv` |

## Installation des bibliothèques

In [ ]:
!pip install requests
!pip install beautifulsoup4
!pip install pandas
!pip install selenium

# Mission 1 : Static Web Scraping avec Requests et BeautifulSoup

### Import needed libraries

In [ ]:
import requests
from bs4 import BeautifulSoup
import pandas as pd
import re

L'objectif est de récupérer le **titre**, le **prix**, la **disponibilité** et la **note** d'au moins 40 livres du site books.toscrape.com, en parcourant plusieurs pages (pagination automatique).

In [ ]:
url = 'https://books.toscrape.com/catalogue/page-1.html'

In [ ]:
req = requests.get(url)

In [ ]:
status = req.status_code
encoding = req.encoding
status

Le statut **200** signifie que la requête a réussi.

Remarque : le serveur n'indique pas l'encodage, donc `requests` choisit `ISO-8859-1` par défaut. Le symbole `£` s'afficherait alors `Â£`. On force donc l'encodage en UTF-8 :

In [ ]:
print(encoding)
req.encoding = 'utf-8'
html_doc = req.text

In [ ]:
html_doc[:1000]

In [ ]:
soup = BeautifulSoup(html_doc, 'html.parser')

In [ ]:
soup

### 1. Analyse de la structure HTML

En inspectant la page (clic droit → *Inspecter*), on observe que :
- chaque livre est dans une balise `<article class="product_pod">` ;
- le **titre** complet est dans l'attribut `title` du lien `<a>` à l'intérieur de `<h3>` (le texte affiché est tronqué : « A Light in the ... ») ;
- le **prix** est dans `<p class="price_color">` (ex. `£51.77`) ;
- la **disponibilité** est dans `<p class="instock availability">` ;
- la **note** est codée dans la classe de `<p class="star-rating Three">` (One, Two, Three, Four, Five) ;
- le bouton **next** est dans `<li class="next"><a href="page-2.html">`.

Chaque page contient 20 livres, il faut donc au moins 2 pages pour 40 livres.

### Localize and collect data

In [ ]:
books = soup.find_all('article', {'class': 'product_pod'})

In [ ]:
len(books)

In [ ]:
books[0]

In [ ]:
for item in books:
    pass

In [ ]:
item

In [ ]:
# book_title : on prend l'attribut 'title' (le texte est tronqué)
title = item.find('h3').find('a').get('title')
title

In [ ]:
# book_price
price = item.find('p', {'class': 'price_color'}).text
price

In [ ]:
# book_availability
availability = item.find('p', {'class': 'instock availability'}).text.strip()
availability

In [ ]:
# book_rating : la note est la 2e classe de la balise
rating = item.find('p', {'class': 'star-rating'}).get('class')
rating

In [ ]:
rating = rating[1]
rating

### Conversion des prix et des notes en valeurs numériques

- **Prix** : on garde uniquement les chiffres et le point (`£51.77` → `51.77`) puis on convertit en `float`.
- **Note** : on utilise un dictionnaire de correspondance (`'Three'` → `3`).

In [ ]:
def convert_price(price_text):
    return float(re.sub(r'[^0-9.]', '', price_text))

convert_price(price)

In [ ]:
rating_map = {'One': 1, 'Two': 2, 'Three': 3, 'Four': 4, 'Five': 5}
rating_map[rating]

In [ ]:
for item in books:
    title = item.find('h3').find('a').get('title')
    price = convert_price(item.find('p', {'class': 'price_color'}).text)
    rating = rating_map[item.find('p', {'class': 'star-rating'}).get('class')[1]]
    print(title + " --> " + str(price) + " £ --> " + str(rating) + " étoiles")

### Gérer la pagination

Au lieu d'écrire les URL à la main, on suit le lien **next** tant qu'il existe et tant qu'on n'a pas atteint le nombre de livres voulu. Le lien est relatif (`page-2.html`), on reconstruit donc l'URL complète avec `urljoin`.

In [ ]:
next_page = soup.find('li', {'class': 'next'})
next_page

In [ ]:
next_page.find('a').get('href')

In [ ]:
from urllib.parse import urljoin
from time import sleep

url = 'https://books.toscrape.com/catalogue/page-1.html'
nb_livres_min = 40
all_books = []
page = 1

while url and len(all_books) < nb_livres_min:
    req = requests.get(url)
    if req.status_code != 200:
        print("Erreur", req.status_code, "sur", url)
        break
    req.encoding = 'utf-8'
    soup = BeautifulSoup(req.text, 'html.parser')

    for item in soup.find_all('article', {'class': 'product_pod'}):
        all_books.append({
            'title': item.find('h3').find('a').get('title'),
            'price': convert_price(item.find('p', {'class': 'price_color'}).text),
            'availability': item.find('p', {'class': 'instock availability'}).text.strip(),
            'rating': rating_map[item.find('p', {'class': 'star-rating'}).get('class')[1]]
        })
    print("Page", page, "traitée -> total :", len(all_books), "livres")

    # page suivante
    next_page = soup.find('li', {'class': 'next'})
    url = urljoin(url, next_page.find('a').get('href')) if next_page else None
    page += 1
    sleep(1)   # on reste poli avec le serveur

print("Nombre total de livres :", len(all_books))

### Organiser les résultats dans un DataFrame Pandas

In [ ]:
df_books = pd.DataFrame(all_books)
df_books.head()

In [ ]:
df_books.info()

In [ ]:
df_books.describe()

### Save data in csv

In [ ]:
df_books.to_csv('books.csv', index=False, encoding='utf-8')
pd.read_csv('books.csv').head()

# Mission 2 : API Explorer avec Requests et l'API REST Open Library

In [ ]:
import requests
import json
import pandas as pd

### 1. Consultation de la documentation

D'après https://openlibrary.org/dev/docs/api/search :
- endpoint : `https://openlibrary.org/search.json`
- paramètres utiles :
  - `q` : la recherche (ici « data science ») ;
  - `fields` : les champs à renvoyer (on limite la réponse à ce qui nous intéresse) ;
  - `limit` : nombre de résultats par page ;
  - `page` : numéro de page.
- la réponse JSON contient `numFound` (nombre total de résultats) et `docs` (la liste des livres).
- champs d'un livre : `title`, `author_name` (liste), `first_publish_year`, `key` (identifiant, ex. `/works/OL...W`).

Comme dans l'exemple de la prof, on passe les paramètres avec `params=` plutôt que de les écrire dans l'URL.

In [ ]:
API = "https://openlibrary.org/search.json"
params = {
    'q': 'data science',
    'fields': 'key,title,author_name,first_publish_year',
    'limit': 20,
    'page': 1
}

In [ ]:
data = requests.get(API, params=params)
data.url

### Vérification du statut HTTP

In [ ]:
data.status_code

In [ ]:
if data.status_code == 200:
    data_json = data.json()
else:
    print("Erreur HTTP :", data.status_code)

In [ ]:
data_json.keys()

In [ ]:
# Nombre total de résultats disponibles
data_json['numFound']

In [ ]:
len(data_json['docs'])

In [ ]:
item = data_json['docs'][0]
item

In [ ]:
print(json.dumps(item, indent=4))

In [ ]:
item.keys()

In [ ]:
item['title']

In [ ]:
item['author_name']

In [ ]:
item['first_publish_year']

In [ ]:
item['key']

### Gestion des champs manquants

Certains livres n'ont pas d'auteur ou pas d'année : `item['author_name']` provoquerait alors une erreur `KeyError`. On utilise `item.get('champ', valeur_par_defaut)` qui renvoie une valeur par défaut si le champ n'existe pas. Les auteurs étant une liste, on les joint en une seule chaîne.

In [ ]:
def extract_book(item):
    authors = item.get('author_name')
    return {
        'id': item.get('key', '').replace('/works/', ''),
        'title': item.get('title'),
        'authors': ', '.join(authors) if authors else None,
        'first_publish_year': item.get('first_publish_year')
    }

extract_book(item)

In [ ]:
# Vérifions combien de livres de la 1re page ont des champs manquants
for doc in data_json['docs']:
    missing = [f for f in ['title', 'author_name', 'first_publish_year'] if f not in doc]
    if missing:
        print(doc.get('title'), '--> champs manquants :', missing)

### Gérer la pagination

Comme dans l'exemple de la World Bank, on récupère la première page puis les suivantes, jusqu'à avoir au moins 30 résultats (ou jusqu'à épuisement des résultats).

In [ ]:
from time import sleep

nb_resultats_min = 30
all_docs = data_json['docs']
nb_total = data_json['numFound']

while len(all_docs) < nb_resultats_min and len(all_docs) < nb_total:
    params['page'] += 1
    response = requests.get(API, params=params)
    if response.status_code != 200:
        print("Erreur HTTP :", response.status_code)
        break
    docs = response.json()['docs']
    if not docs:        # plus de résultats
        break
    all_docs.extend(docs)
    sleep(1)

print("Nombre de résultats collectés :", len(all_docs))

In [ ]:
for doc in all_docs[:10]:
    print(str(doc.get('title')) + ' --> ' + str(doc.get('first_publish_year')))

### Organiser les données dans un DataFrame Pandas

In [ ]:
df_ol = pd.DataFrame([extract_book(doc) for doc in all_docs])
# année en entier "nullable" (accepte les valeurs manquantes)
df_ol['first_publish_year'] = df_ol['first_publish_year'].astype('Int64')
df_ol.head()

In [ ]:
df_ol.info()

In [ ]:
# Nombre de valeurs manquantes par colonne
df_ol.isna().sum()

### Save data in csv

In [ ]:
df_ol.to_csv('openlibrary.csv', index=False, encoding='utf-8')
pd.read_csv('openlibrary.csv').head()

# Mission 3 : Dynamic Web Scraping avec Selenium

Sur https://quotes.toscrape.com/js/, les citations sont **générées par JavaScript** : elles ne sont pas dans le HTML renvoyé par le serveur. Vérifions-le avec Requests + BeautifulSoup :

In [ ]:
import requests
from bs4 import BeautifulSoup

req = requests.get('https://quotes.toscrape.com/js/')
soup = BeautifulSoup(req.text, 'html.parser')
len(soup.find_all('div', {'class': 'quote'}))

Résultat : **0** citation trouvée → il faut un vrai navigateur qui exécute le JavaScript, d'où **Selenium**.

Remarque : avec Selenium 4, il n'est plus nécessaire de donner `executable_path` (l'argument utilisé dans l'exemple de la prof n'existe plus) : *Selenium Manager* télécharge automatiquement le bon driver pour Chrome.

### Import

In [ ]:
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
import pandas as pd

### 1. Ouvrir le site avec Selenium

In [ ]:
driver = webdriver.Chrome()
driver.get("https://quotes.toscrape.com/js/")

### 2. Attendre le chargement des citations

Au lieu d'un `sleep(10)` fixe comme dans l'exemple eBay, on utilise **WebDriverWait** : il attend (au maximum 10 s) que les éléments `div.quote` soient présents, et continue dès qu'ils le sont.

In [ ]:
wait = WebDriverWait(driver, 10)
wait.until(EC.presence_of_all_elements_located((By.CLASS_NAME, "quote")))

### Localise data

En inspectant la page :
- chaque citation est dans `<div class="quote">` ;
- le texte dans `<span class="text">` ;
- l'auteur dans `<small class="author">` ;
- les tags dans des liens `<a class="tag">` à l'intérieur de `<div class="tags">` ;
- le bouton suivant dans `<li class="next"><a>`.

In [ ]:
results = driver.find_elements(By.CLASS_NAME, 'quote')

In [ ]:
len(results)

In [ ]:
for quote in results:
    pass

In [ ]:
quote.text

In [ ]:
# Get the text of the quote
text = quote.find_element(By.CLASS_NAME, "text").text
print("Citation : ", text)

In [ ]:
# Get the author
author = quote.find_element(By.CLASS_NAME, "author").text
print("Auteur : ", author)

In [ ]:
# Get the tags (plusieurs éléments -> find_elements)
tags = [t.text for t in quote.find_elements(By.CLASS_NAME, "tag")]
print("Tags : ", tags)

### Interacting with the page : Clicking Next Page Using Selenium

Pour les deux premières pages, on :
1. attend les citations avec WebDriverWait ;
2. extrait les données ;
3. clique sur **Next** et attend que l'ancienne page disparaisse (`staleness_of`) pour être sûr de ne pas relire la même page.

Le bloc `try / finally` garantit que le navigateur est **fermé correctement** avec `driver.quit()`, même si une erreur survient.

In [ ]:
driver = webdriver.Chrome()
all_quotes = []
nb_pages = 2

try:
    driver.get("https://quotes.toscrape.com/js/")
    wait = WebDriverWait(driver, 10)

    for page in range(1, nb_pages + 1):
        # attendre le chargement des citations
        wait.until(EC.presence_of_all_elements_located((By.CLASS_NAME, "quote")))
        results = driver.find_elements(By.CLASS_NAME, "quote")

        for quote in results:
            all_quotes.append({
                'quote': quote.find_element(By.CLASS_NAME, "text").text,
                'author': quote.find_element(By.CLASS_NAME, "author").text,
                'tags': ', '.join(t.text for t in quote.find_elements(By.CLASS_NAME, "tag")),
                'page': page
            })
        print("Page", page, ":", len(results), "citations")

        # aller à la page suivante (sauf après la dernière page voulue)
        if page < nb_pages:
            next_page = wait.until(EC.element_to_be_clickable((By.CSS_SELECTOR, "li.next a")))
            first_quote = results[0]
            next_page.click()
            wait.until(EC.staleness_of(first_quote))
finally:
    driver.quit()

print("Nombre total de citations :", len(all_quotes))

### Organiser les données dans un DataFrame et Save data in csv

In [ ]:
df_quotes = pd.DataFrame(all_quotes)
df_quotes.head()

In [ ]:
df_quotes.info()

In [ ]:
df_quotes.to_csv('quotes.csv', index=False, encoding='utf-8')
pd.read_csv('quotes.csv').head()

# Comparaison des trois méthodes

| Critère | Requests + BeautifulSoup | API REST | Selenium |
|---|---|---|---|
| Type de contenu | HTML statique | JSON structuré | HTML généré par JavaScript |
| Vitesse | Rapide | Très rapide | Lente (lance un navigateur) |
| Difficulté | Moyenne (analyser le HTML) | Faible (lire la documentation) | Plus élevée (driver, attentes) |
| Robustesse | Fragile si le HTML change | Stable (format documenté) | Fragile si le HTML change |
| Données | À nettoyer (`£51.77`, `"Three"`) | Déjà typées, mais champs manquants | À nettoyer (guillemets, tags) |
| Pagination | Suivre le lien *next* | Paramètre `page` | Cliquer sur *Next* |
| Quand l'utiliser | Pages statiques sans API | Dès qu'une API existe | Contenu dynamique / interactions |

**Conclusion :** l'API est la méthode à privilégier quand elle existe ; BeautifulSoup convient aux pages statiques ; Selenium n'est utile que lorsque le contenu dépend du JavaScript ou d'interactions (clics, formulaires).

# Question de conclusion

### Les données extraites sont-elles directement exploitables pour le Data Mining ?

**Non, pas entièrement.** Elles sont structurées en tableaux, mais elles présentent encore des défauts :
- `books.csv` : la disponibilité est un texte (« In stock ») et non une variable binaire ; on n'a pas la quantité en stock ;
- `openlibrary.csv` : des valeurs manquantes (auteurs, années), des doublons possibles (même livre en plusieurs éditions), plusieurs auteurs dans une même cellule ;
- `quotes.csv` : les citations contiennent des guillemets typographiques (“ ”), les tags sont regroupés dans une seule chaîne de caractères.

### Trois opérations de prétraitement

1. **Gestion des valeurs manquantes et des doublons** : supprimer les doublons (`drop_duplicates`), puis supprimer ou imputer les valeurs manquantes (ex. année de publication manquante dans Open Library).
2. **Nettoyage et transformation des types** : convertir la disponibilité en variable binaire (1/0), enlever les guillemets et espaces des citations, normaliser le texte (minuscules, accents), vérifier que prix, notes et années sont bien numériques.
3. **Encodage et normalisation** : transformer les variables catégorielles ou multi-valeurs en variables numériques (ex. *one-hot encoding* des tags ou des auteurs), et normaliser les variables numériques (ex. prix avec une normalisation Min-Max) pour que les algorithmes ne soient pas biaisés par les échelles.

In [ ]:
# Exemple d'application de ces prétraitements
df_books = pd.read_csv('books.csv')
df_books['in_stock'] = df_books['availability'].str.contains('In stock').astype(int)
df_books['price_norm'] = (df_books['price'] - df_books['price'].min()) / (df_books['price'].max() - df_books['price'].min())

df_ol = pd.read_csv('openlibrary.csv').drop_duplicates(subset='id')
print("Valeurs manquantes Open Library :\n", df_ol.isna().sum())

df_quotes = pd.read_csv('quotes.csv')
df_quotes['quote'] = df_quotes['quote'].str.strip('“”')
tags_dummies = df_quotes['tags'].fillna('').str.get_dummies(sep=', ')

df_books.head()